# 01 — Build cell-type prototypes

Reads scRNA-seq data from a `.h5ad` file and produces per-cell-type average gene
expression profiles (prototypes) restricted to genes that are also in the Xenium panel.

**Inputs**
- `data/pancreas.h5ad` — AnnData file containing the scRNA-seq expression matrix and
  cell metadata (both live inside this single file)
- `data/transcripts.parquet` — Xenium transcript table; used to derive the Xenium gene panel

**Outputs**
- `data/prototypes.csv` — mean expression per cell type (cell_types × genes)
- `data/prototypes_normalized.csv` — L2-normalised version for cosine similarity
- `data/shared_genes.txt` — genes present in both the scRNA-seq data and the Xenium panel

In [30]:
import numpy as np
import pandas as pd
import anndata as ad
import pyarrow.parquet as pq
from sklearn.preprocessing import normalize

## Config

In [31]:
H5AD_PATH            = "../data/pancreas.h5ad"
TRANSCRIPT_PATH      = "../data/transcripts.parquet"
PROTOTYPES_PATH      = "../data/prototypes.csv"
PROTOTYPES_NORM_PATH = "../data/prototypes_normalized.csv"
SHARED_GENES_PATH    = "../data/shared_genes.txt"

# Column in adata.obs that holds cell type labels.
# Run the inspection cell below first if you're unsure what this should be.
CELL_TYPE_COL = "celltype"

# Column in transcripts.parquet that holds gene names.
# Common options: 'feature_name', 'gene_name'
GENE_COL = "feature_name"

## 1 · Load and inspect the .h5ad file

An `.h5ad` file is an [AnnData](https://anndata.readthedocs.io) object.
- `adata.X` — expression matrix (cells × genes), may be a sparse matrix
- `adata.obs` — cell metadata dataframe (one row per cell)
- `adata.var` — gene metadata dataframe (one row per gene, index = gene names)

In [32]:
adata = ad.read_h5ad(H5AD_PATH)
print(adata)
print("\nCell metadata columns (adata.obs):")
print(adata.obs.columns.tolist())
print("\nFirst few rows of adata.obs:")
print(adata.obs.head())

AnnData object with n_obs × n_vars = 14693 × 2448
    obs: 'celltype', 'sample', 'n_genes', 'batch', 'n_counts', 'louvain'
    var: 'n_cells-0', 'n_cells-1', 'n_cells-2', 'n_cells-3'
    uns: 'celltype_colors', 'louvain', 'neighbors', 'pca', 'sample_colors'
    obsm: 'X_pca', 'X_umap'
    varm: 'PCs'
    obsp: 'connectivities', 'distances'

Cell metadata columns (adata.obs):
['celltype', 'sample', 'n_genes', 'batch', 'n_counts', 'louvain']

First few rows of adata.obs:
                              celltype sample  n_genes batch  n_counts louvain
index                                                                         
human1_lib1.final_cell_0001-0   acinar  Baron     3526     0   22411.0       2
human1_lib1.final_cell_0002-0   acinar  Baron     4201     0   27949.0       2
human1_lib1.final_cell_0003-0   acinar  Baron     2119     0   16892.0       2
human1_lib1.final_cell_0004-0   acinar  Baron     2956     0   19299.0       2
human1_lib1.final_cell_0005-0   acinar  Baron     27

In [33]:
assert CELL_TYPE_COL in adata.obs.columns, (
    f"'{CELL_TYPE_COL}' not found in adata.obs. "
    f"Available columns: {adata.obs.columns.tolist()}"
)
print("Cell types found:")
print(adata.obs[CELL_TYPE_COL].value_counts())

Cell types found:
celltype
alpha                     4214
beta                      3354
ductal                    1804
acinar                    1368
not applicable            1154
delta                      917
gamma                      571
endothelial                289
activated_stellate         284
dropped                    178
quiescent_stellate         173
mesenchymal                 80
macrophage                  55
PSC                         54
unclassified endocrine      41
co-expression               39
mast                        32
epsilon                     28
mesenchyme                  27
schwann                     13
t_cell                       7
MHC class II                 5
unclear                      4
unclassified                 2
Name: count, dtype: int64


## 2 · Extract expression matrix and gene names

In [34]:
import scipy.sparse as sp

# adata.X may be sparse — convert to dense DataFrame
X = adata.X
if sp.issparse(X):
    X = X.toarray()

expr = pd.DataFrame(X, index=adata.obs_names, columns=adata.var_names)
print("Expression matrix:", expr.shape, "(cells × genes)")

Expression matrix: (14693, 2448) (cells × genes)


## 3 · Find shared genes with the Xenium panel

We derive the Xenium gene panel directly from `transcripts.parquet`
so there's no separate gene panel file to maintain.

In [35]:
tx = pd.read_parquet(TRANSCRIPT_PATH, columns=[GENE_COL], engine="fastparquet")
xenium_genes = tx[GENE_COL].unique().tolist()

shared_genes = sorted(set(expr.columns) & set(xenium_genes))
print(f"scRNA-seq genes : {len(expr.columns):,}")
print(f"Xenium panel    : {len(xenium_genes):,}")
print(f"Shared genes    : {len(shared_genes):,}")

expr = expr[shared_genes]

with open(SHARED_GENES_PATH, "w") as f:
    f.write("\n".join(shared_genes))
print(f"\nSaved shared genes → {SHARED_GENES_PATH}")

scRNA-seq genes : 2,448
Xenium panel    : 541
Shared genes    : 126

Saved shared genes → ../data/shared_genes.txt


## 4 · Build prototypes by averaging per cell type

In [36]:
cell_types = adata.obs[CELL_TYPE_COL]
prototypes = expr.groupby(cell_types).mean()
print("Prototype matrix:", prototypes.shape, "(cell_types × genes)")
print(prototypes.index.tolist())

Prototype matrix: (24, 126) (cell_types × genes)
['MHC class II', 'PSC', 'acinar', 'activated_stellate', 'alpha', 'beta', 'co-expression', 'delta', 'dropped', 'ductal', 'endothelial', 'epsilon', 'gamma', 'macrophage', 'mast', 'mesenchymal', 'mesenchyme', 'not applicable', 'quiescent_stellate', 'schwann', 't_cell', 'unclassified', 'unclassified endocrine', 'unclear']


## 4 · L2-normalise for cosine similarity

In [37]:
proto_norm = pd.DataFrame(
    normalize(prototypes.values, norm="l2"),
    index=prototypes.index,
    columns=prototypes.columns
)

## 5 · Save

In [38]:
prototypes.to_csv(PROTOTYPES_PATH)
proto_norm.to_csv(PROTOTYPES_NORM_PATH)
print(f"Saved prototypes          → {PROTOTYPES_PATH}")
print(f"Saved prototypes_normalized → {PROTOTYPES_NORM_PATH}")

Saved prototypes          → ../data/prototypes.csv
Saved prototypes_normalized → ../data/prototypes_normalized.csv
